# **LangChain 구성요소 - 모델 호출하기**

## 실습 소개

이번 실습에서는 LangChain을 활용하여 **LLM을 직접 호출하고 응답을 받아보는 방법**을 배웁니다. LangChain은 다양한 언어 모델을 동일한 방식으로 다룰 수 있도록 인터페이스를 통합하고, 반복적인 호출 로직을 간단하게 만들어 줍니다.

## 학습 목표

- 언어 모델 객체 생성
- 프롬프트와 결합하여 모델 호출
- 모델로부터 생성된 응답 출력

## 모델 호출하기
LangChain은 OpenAI의 ChatGPT를 비롯하여 HuggingFace, Google 등 다양한 모델을 쉽게 호출할 수 있습니다. 그 중에서 ChatOpenAI를 이용하면 OpenAI의 ChatGPT를 호출할 수 있습니다. 

### OpenAI API KEY 등록하기
ChatGPT를 사용하기 위해서는 OpenAI API KEY가 있어야 합니다. OpenAI API KEY를 등록하는 방법은 다음과 같습니다. `None` 위치에 "sk-****" 를 작성하고 해당 코드를 실행하면 됩니다. 

(이후의 실습은 API KEY가 있다는 가정 하에 진행됩니다.)

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = None

OpenAI API 키를 `.env` 파일을 통해 관리하는 방법은 아래와 같습니다. (`.env` 파일이 실행 경로에 저장되어있다고 가정합니다.)

In [ ]:
from dotenv import load_dotenv
load_dotenv()

### ChatGPT 모델 사용하기

#### `temperature`

ChatOpenAI를 사용하면 OpenAI의 대화형 언어 모델을 사용할 수 있고 옵션을 조정할 수 있습니다. 그 중에서 대표적인 옵션인 `temperature` 옵션은 모델의 창의성(다양성)에 대한 옵션을 의미하며 0으로 설정하면 더욱 일관된 답변을 생성합니다. (범위는 0 ~ 1 사이)
 
`temperature=0.0`은 보수적이고 예측 가능한 응답을 생성하고, `temperature=1.0`에 가까울수록 창의적이고 다양한 응답이 생성됩니다. 일반적으로 0.3 ~ 0.7 정도가 안정적인 응답 품질을 보장하는 범위입니다.

#### `model` 선택 기준

LangChain에서는 OpenAI의 다양한 언어 모델을 `model_name`으로 지정할 수 있습니다. 실제 서비스에서는 **응답 품질과 비용의 균형**을 고려하여 모델을 선택합니다.

| 모델 이름           | 주요 특징                                           | 추천 사용 사례                            |
| --------------- | ----------------------------------------------- | ----------------------------------- |
| `gpt-4o`        | 텍스트, 이미지, 오디오 입력을 실시간으로 처리하는 멀티모달 모델. 빠르고 효율적임. | 실시간 대화, 멀티모달 응용 프로그램, 고급 사용자 인터페이스  |
| `gpt-4.1-mini`  | GPT-4.1 기반의 경량화 모델. 빠른 응답과 낮은 비용을 제공함.        | 비용 효율적인 애플리케이션, 빠른 응답이 필요한 서비스      |
| `o1` | 고급 추론 모델. 문제 해결 전 내부적으로 ‘사고 흐름’을 먼저 생성함. 사용 비용이 높음.             | 수학, 과학, 코딩 등 단계적 사고가 필요한 고정밀 응용 분야 |



OpenAI의 언어 모델은 종류가 다양하고 훈련 시점, 사용량에 따른 비용이 다르기 때문에 [OpenAI 홈페이지](https://platform.openai.com/docs/models)에서 모델과 비용을 꼭 확인 후 사용하세요.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model_name = "gpt-4.1-mini",
    temperature = 0
)

모델 설정이 마무리되었다면 프롬프트(chat_prompt)를 이용하여 답변을 확인해 보겠습니다. 

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt_template = ChatPromptTemplate(
    [
        (
            "system",
            "당신은 한국 역사에 대해 전문적인 지식을 가진 전문 AI 어시스턴트입니다. 이름은 {name}입니다.",
        ),
        ("human", "안녕하세요 저는 {history_keyword}에 대해 궁금해요!"),
        ("ai", "같이 한 번 알아볼까요? 어떤게 궁금하신가요?"),
        ("human", "{question}"),
    ]
)

chat_prompt = chat_prompt_template.format_messages(
    name="헬로빗", history_keyword="조선", question="세종대왕 님의 음식 취향이 궁금해요"
)

In [ ]:
# 현재 프롬프트 확인하기
chat_prompt

In [ ]:
answer = llm.invoke(chat_prompt)
answer

AIMessage는 답변을 포함한 다양한 정보를 보여주기 때문에 답변(content)만 확인하겠습니다.

In [ ]:
answer.content

위의 코드를 이용하여 모델에게 자유롭게 질문해 보세요.

In [ ]:
None

마지막으로 영화 리뷰 데이터를 활용하여 감정 분류를 확인해 보겠습니다.

In [ ]:
from langchain_core.prompts import PromptTemplate

# 리뷰 데이터
review_list = ["너무 난해하고 복잡한 영화였어요", "너무 재미있었어요!", "지루해요"]

# 프롬프트 생성하기
template = "문구 : {text} \n문구를 보고 {result1} 또는 {result2} 중에 하나로 분류해 줘, 답변은 {result1} 또는 {result2} 으로만 대답해 줘"

prompt = PromptTemplate(
    input_variables = {"text", "result1", "result2"},
    template = template
)

for review_txt in review_list :
    
    prompt_txt = prompt.format(
        text = review_txt,
        result1 = "긍정",
        result2 = "부정"
    )

    # 모델의 분류 결과 확인하기
    print(review_txt)
    print(">>", llm.invoke(prompt_txt).content)

### LangChain을 꼭 사용해서 모델을 호출해야하나요?

일반적으로 OpenAI API를 호출하려면 `chat.completions.create()` 같은 직접적인 API 호출을 사용해야 합니다.
#### LangChain 없이 직접 모델 호출

In [ ]:
from openai import OpenAI

# 모델 호출
client = OpenAI()

completion = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "user", "content": "LangChain이 뭐야?"}],
    temperature=0.5,
    
)

# 응답 출력
print(completion.choices[0].message.content)

이렇게 직접 호출하면 요청 형식, messages 배열, 모델 명 등 직접 지정해야 하는 번거로움이 있습니다. 또한 프롬프트 재사용하기가 어렵고 복잡한 입력일수록 코드가 지저분해지고 반복된다는 단점이 있습니다.

#### LangChain을 사용하여 모델 호출

In [ ]:
# 모델 객체 생성
llm = ChatOpenAI(model_name="gpt-4.1-mini", temperature=0.5)

# 프롬프트 입력
response = llm.invoke("LangChain이 뭐야?")
print(response.content)

LangChain을 사용한 경우 문자열 그대로 프롬프트를 전달할 수 있고  추후 PromptTemplate, Chain, Memory 등과 자연스럽게 연동 가능합니다. 또한 코드가 간결하고 재사용에 유리하기 때문에 LangChain을 활용한 방법이 더 효율적입니다.